# ChatIR web app on Colab, with a shareable link

Runs the human-in-the-loop web app (`webapp/`) on this Colab GPU and gives you a public
`https://….trycloudflare.com` link anyone can open. People describe an image, the PlugIR agent
asks questions, and their answers steer the search over the 50k ChatIR corpus.

* **Agent:** PlugIR baseline (`configs/experiments/plugir.yaml`) with the team's `qwen3.5:9b`, served by Ollama on this GPU. Free, no API key.
* **Retriever:** frozen BLIP over the 50k ChatIR search space. The index is built from your image embeddings (`corpus_blip_large.pth`), so no images are embedded here.
* **Images:** shown straight from COCO's servers, so the 19 GB image set is **not** downloaded.

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then *Run all*. The link is printed by section 5 (about 10 minutes the first time).

**The link only works while this notebook is running.** Free Colab sessions end after a few hours, or sooner if the tab is left idle. Each new session gets a new link.

## 1. Settings

In [ ]:
BRANCH = 'feat/interactive-webapp'   # switch to 'main' once the web app is merged
MODEL = 'qwen3.5:9b'                 # team-pinned questioner (configs/base.yaml)
EXPERIMENT = 'plugir'                # any configs/experiments/<name>.yaml, e.g. 'divergence'

# Google Drive keeps the index and everyone's contributions between sessions.
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/chatir-cornell2026'
CORPUS_FILE = 'corpus_blip_large.pth'   # PlugIR-format (ids, embeddings) from the PlugIR notebook

## 2. Install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
!git clone -q -b {BRANCH} https://github.com/nopharshalom/ChatIR-Cornell2026 2>/dev/null || (cd ChatIR-Cornell2026 && git fetch -q && git checkout -q {BRANCH} && git pull -q)
%cd /content/ChatIR-Cornell2026
!pip install -q -e ".[serve]"

## 3. Retrieval index

Looks for, in order: an index saved to Drive by an earlier session, then your `corpus_blip_large.pth`
(in the Drive folder, its `cache/`, or `/content`), and otherwise asks you to upload that file.
Building from it only encodes the 50k captions (about a minute on the GPU).

In [ ]:
import os, shutil
B = '-c configs/base.yaml -c configs/backends/colab.yaml'
os.makedirs('cache', exist_ok=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_DIR}/cache', exist_ok=True)

saved_index = f'{DRIVE_DIR}/cache/index_blip.pt'
if os.path.exists('cache/index_blip.pt'):
    print('index already here')
elif USE_DRIVE and os.path.exists(saved_index):
    shutil.copy(saved_index, 'cache/index_blip.pt'); print('index copied from Drive')
else:
    candidates = [f'cache/{CORPUS_FILE}', f'{DRIVE_DIR}/{CORPUS_FILE}', f'{DRIVE_DIR}/cache/{CORPUS_FILE}',
                  f'/content/{CORPUS_FILE}', f'/content/cache/{CORPUS_FILE}']
    src = next((p for p in candidates if os.path.exists(p)), None)
    if src is None:
        from google.colab import files
        print(f'Upload {CORPUS_FILE} (the (ids, embeddings) file from the PlugIR notebook):')
        name = next(iter(files.upload()))
        src = f'cache/{name}'
        shutil.move(name, src)
    print('building index from', src)
    !agentir build-index {B} --from-plugir-cache "{src}"
    if USE_DRIVE:
        shutil.copy('cache/index_blip.pt', saved_index); print('saved index to Drive for next time')

## 4. Start Qwen3.5-9B (Ollama, on this GPU)

In [ ]:
import subprocess, time, urllib.request
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -1 /tmp/ollama_install.log

env = dict(os.environ, OLLAMA_NUM_PARALLEL='4', OLLAMA_CONTEXT_LENGTH='8192', OLLAMA_KEEP_ALIVE='-1')
ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/content/ollama.log', 'w'),
                               stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2); print('ollama is up'); break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL} 2>&1 | tail -1

## 5. Start the web app and the public link

Contributions (every question, answer and query) go to `contributions.jsonl` in the Drive folder
when `USE_DRIVE` is on, so they survive the session. Rerun this cell to restart the app (the link changes).

In [ ]:
import re, json
if USE_DRIVE:
    os.makedirs(f'{DRIVE_DIR}/webapp', exist_ok=True)
    log_path = f'{DRIVE_DIR}/webapp/contributions.jsonl'
else:
    log_path = 'runs/webapp/contributions.jsonl'

for p in ('app_proc', 'tunnel_proc'):   # rerunning this cell restarts both
    if p in globals():
        globals()[p].terminate()

app_proc = subprocess.Popen(
    ['python', '-m', 'webapp.server', *B.split(), '-c', f'configs/experiments/{EXPERIMENT}.yaml',
     '--set', f'agent.llm.model={MODEL}', '--log', log_path, '--host', '0.0.0.0', '--port', '8000'],
    stdout=open('/content/webapp.log', 'w'), stderr=subprocess.STDOUT)
info = None
for _ in range(300):   # loading BLIP takes ~1 min
    try:
        info = json.load(urllib.request.urlopen('http://localhost:8000/api/info', timeout=2)); break
    except Exception:
        if app_proc.poll() is not None:
            raise RuntimeError('web app exited; see /content/webapp.log:\n' + open('/content/webapp.log').read()[-2000:])
        time.sleep(1)
print('web app up:', info['backend'], '| corpus', info['corpus_size'], '| model', info['method'].get('llm'))

if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel_proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000'],
                               stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        url = m.group(0); break
    time.sleep(1)
from IPython.display import HTML, display
display(HTML(f'<h2>Share this link: <a href="{url}" target="_blank">{url}</a></h2>') if url
        else HTML('<b>No tunnel URL yet; see /content/tunnel.log</b>'))

## 6. While it's running

* **Keep this tab open.** Colab stops idle sessions, and the link dies with the session.
* **Anyone with the link can use it.** There is no login, so only share it with the people you want using your Colab GPU.
* The first question takes longest while the models warm up. After that a PlugIR round is roughly 10–30 s on a T4.
* To demo another method, change `EXPERIMENT` in section 1 and rerun section 5.

In [ ]:
# Live status: recent server log and how many dialogs people have finished
!tail -n 15 /content/webapp.log
n = sum(1 for l in open(log_path) if '"event": "finish"' in l) if os.path.exists(log_path) else 0
print(f'\nfinished dialogs so far: {n}  ({log_path})')

In [ ]:
# Download everyone's contributions (also kept on Drive when USE_DRIVE is on)
from google.colab import files
files.download(log_path)